# Persistence and Checkpointers in LangGraph

LangGraph provides built-in persistence through **checkpointers**. A checkpointer saves the state of your graph after every step, allowing you to:

- Resume execution from where you left off
- Maintain conversation history across invocations
- Inspect and travel through past states (time-travel debugging)
- Build long-running stateful agents

In [ ]:
# %pip install langgraph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
from langchain_openai import ChatOpenAI

## InMemorySaver (MemorySaver)

The `MemorySaver` keeps checkpoints in memory. It is the simplest checkpointer and is perfect for **development, testing, and tutorials**.

**Do not use this in production** — checkpoints are lost when the process exits, and the in-memory dictionary does not scale across machines.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class State(TypedDict):
    messages: list
    count: int

def increment(state: State):
    return {"count": state["count"] + 1}

memory = MemorySaver()
builder = StateGraph(State)
builder.add_node("increment", increment)
builder.add_edge(START, "increment")
builder.add_edge("increment", END)
graph = builder.compile(checkpointer=memory)

# Thread 1
config1 = {"configurable": {"thread_id": "thread-1"}}
result = graph.invoke({"messages": [], "count": 0}, config=config1)
print(result)

# Resume same thread
result2 = graph.invoke({"messages": [], "count": 0}, config=config1)
print("After resume:", result2)

## The `thread_id` config pattern

Every call to a checkpointed graph requires a `thread_id` inside the `configurable` config dict. The `thread_id` is the identifier LangGraph uses to look up (and write back) the state for that conversation.

- **Same `thread_id`** → continues an existing conversation, state is reloaded
- **New `thread_id`** → starts a fresh conversation with a clean state

This is how you support multiple parallel users or conversations on the same graph.

In [ ]:
# Two independent threads on the same graph
config_a = {"configurable": {"thread_id": "thread-1"}}
config_b = {"configurable": {"thread_id": "thread-2"}}

# Invoke thread-1 a few times
graph.invoke({"messages": [], "count": 0}, config=config_a)
graph.invoke({"messages": [], "count": 0}, config=config_a)
state_a = graph.get_state(config_a)

# Invoke thread-2 once
graph.invoke({"messages": [], "count": 0}, config=config_b)
state_b = graph.get_state(config_b)

print("thread-1 count:", state_a.values["count"])
print("thread-2 count:", state_b.values["count"])

## SqliteSaver

The `SqliteSaver` persists checkpoints to a SQLite database. Unlike `MemorySaver`, state **survives process restarts** — making it a good choice for local development, single-node deployments, and small production workloads.

Install the optional dependency first:

```bash
pip install langgraph-checkpoint-sqlite
```

In [ ]:
# pip install langgraph-checkpoint-sqlite
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string(":memory:") as checkpointer:
    graph = builder.compile(checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "sqlite-thread-1"}}
    result = graph.invoke({"messages": [], "count": 0}, config=config)
    print(result)

## Inspecting State

Every checkpointed graph exposes two introspection methods:

- `graph.get_state(config)` — returns the **current** state snapshot for a thread
- `graph.get_state_history(config)` — returns an iterable of **all** historical checkpoints, newest first

This is the foundation for time-travel debugging: you can replay a thread from any past checkpoint.

In [ ]:
# Rebuild with MemorySaver for inspection examples
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "thread-1"}}
graph.invoke({"messages": [], "count": 0}, config=config)
graph.invoke({"messages": [], "count": 0}, config=config)

# Get current state of a thread
state = graph.get_state(config)
print("Current state:", state)

# Get full history
history = list(graph.get_state_history(config))
print(f"History has {len(history)} checkpoints")
for s in history:
    print(s.config, s.values)

## Updating State

Sometimes you need to manually modify the state of a thread — for example, to inject a correction or override a value before resuming execution. Use `graph.update_state(config, values)` for this.

In [ ]:
# Manually overwrite a value on the thread
graph.update_state(config, {"count": 99})

state = graph.get_state(config)
print("After manual update:", state.values)

## Cross-thread Store

Checkpointers are **scoped to a single thread** — they cannot share data between conversations.

For long-term memory that needs to be shared across threads (e.g. user profile, learned preferences), LangGraph provides a separate `Store` abstraction. The simplest implementation is `InMemoryStore`.

In [ ]:
from langgraph.store.memory import InMemoryStore
from langgraph.graph import StateGraph, START, END

store = InMemoryStore()

# Store something
user_id = "user-123"
store.put(("users", user_id), "profile", {"name": "Lucas", "topic": "LangGraph"})

# Retrieve it
item = store.get(("users", user_id), "profile")
print(item.value)

## Production Note

For production deployments, use `PostgresSaver` from the `langgraph-checkpoint-postgres` package. Postgres handles concurrency, durability, and horizontal scaling far better than SQLite or in-memory checkpointers.

See the official reference for the full list of supported checkpointers and stores:

https://langchain-ai.github.io/langgraph/reference/checkpoints/